# React — Testing

> **Reminder of the three places.** This notebook runs plain JavaScript on the Deno kernel.
> `playground/` is where you see real React behave. Mini-projects are where you build.
> Blocks are labelled **Runnable — plain JS**, **The React API**, **In the playground** or
> **In your project**.
>
> **Where this topic happens.** Not in the shared playground — that project is deliberately React
> and Vite only, and a test runner is a dependency. All the work here happens in
> **`react-scratch`**, the throwaway project you created in LESSON 2 and have used for topics 01,
> 04 and 20. Nothing you install here touches the playground or your finished mini-projects.
>
> The versions below were installed and run while writing these lessons. Install with `@latest`
> and expect newer numbers; the commands are what matter.

## LESSON 84 — What is worth testing, and setting up Vitest

Most bad experiences with testing come from writing the wrong tests, not from writing too few.
So this lesson is half judgement and half setup.

### The test that is worth writing

Ask one question: **if this broke, would I find out?** A test earns its place when the answer is
"not until a user tells me".

| worth testing | why |
|---|---|
| pure logic — reducers, validators, selectors, formatters | rules with many branches, easy to get subtly wrong, and free to test |
| a component's **behaviour** — what appears after a click | it is what the user experiences |
| bug fixes | write the test that fails first, then fix it; the bug cannot come back silently |
| the four states of a fetch (L46) | the empty and error branches are the ones nobody clicks through by hand |

And the ones that cost more than they give:

| not worth it | why |
|---|---|
| implementation details — state variable names, which hook was called | they change when you refactor working code, and the test fails while the app still works |
| trivial rendering — "the heading says Projects" | you will not ship that broken, and the test breaks on every copy edit |
| third-party libraries | React Router's own tests are not your job |
| exact markup or class names | the test now fails when the design changes, which is not a failure |

The line between the two tables is the whole subject: **test what the component does, not how it
does it.** A test that has to be rewritten every time you refactor is not protecting you — it is
duplicating the code in a second, worse language.

### Vitest, because you already have Vite

> Vitest (pronounced as *"veetest"*) is a next generation testing framework powered by Vite.

The practical consequence is that it reads your existing `vite.config.*`, so your plugins,
aliases and JSX handling already work — there is no second build to configure. In a Vite project,
Vitest is the default choice for that reason alone.

**In your project — `react-scratch`:**

```bash
npm install -D vitest jsdom @testing-library/react @testing-library/user-event @testing-library/jest-dom
```

The versions that produced everything in these lessons: `vitest 5.0.0`, `jsdom 30.0.1`,
`@testing-library/react 16.3.3`, `@testing-library/user-event 14.6.7`,
`@testing-library/jest-dom 7.0.1`, on Vite 8.3.0 and React 19.3.0.

Add the script:

```json
{ "scripts": { "test": "vitest" } }
```

`npm test` starts it in watch mode; `npx vitest run` runs once and exits, which is what a CI
server wants.

Then the test settings, in the `vite.config.js` you already have:

```js
import { defineConfig } from "vite";
import react from "@vitejs/plugin-react";

export default defineConfig({
  plugins: [react()],
  test: {
    environment: "jsdom",              // a fake DOM, because Node has none
    globals: true,                     // describe/it/expect without importing them
    setupFiles: "./src/setupTests.js", // runs before every test file
  },
});
```

```js
// src/setupTests.js
import "@testing-library/jest-dom/vitest";
```

Four decisions worth understanding rather than copying:

- **`environment: "jsdom"`** — tests run in Node, which has no `document`. jsdom provides one.
  Leave it out and `render()` fails immediately.
- **`globals: true`** is optional. Without it you write `import { describe, it, expect } from
  "vitest"` in each file, which is more explicit and works identically. Both are shown below.
- **`setupFiles`** runs once per test file. `@testing-library/jest-dom/vitest` adds the matchers
  that make assertions readable — `toBeInTheDocument`, `toHaveTextContent`.
- **No separate config file.** Vitest reads `vite.config.js`; a `vitest.config.js` is only needed
  when the test setup has to diverge from the build.

### What a test file looks like

```js
// src/validate.test.js — the name pattern Vitest picks up
import { describe, expect, it } from "vitest";
import { validate } from "./validate.js";

describe("validate", () => {
  it("requires a name", () => {
    expect(validate({ name: "", email: "a@b" })).toEqual({ name: "Name is required" });
  });
});
```

`describe` groups, `it` (or `test`) is one case, `expect(...)` makes the claim. `toEqual`
compares structurally; `toBe` compares with `Object.is` — LESSON 40's distinction, in a new place.

### Key Notes

- Test what would otherwise reach a user: pure logic, behaviour, and every bug you fix.
- Do not test implementation details, markup, or other people's libraries.
- Vitest reads your existing `vite.config.js`; add a `test` block with `environment: "jsdom"`.
- `npm test` watches; `npx vitest run` runs once.

### Example

**Runnable — plain JS.** A test runner is not magic, and seeing that removes any mystery about
what `it` and `expect` are. This is roughly what Vitest does, minus the parts you would rather not
write yourself.

In [ ]:
// L84 — a test runner in twenty lines

const l84Results = [];

function l84It(name, body) {
  try {
    body();
    l84Results.push({ name, ok: true });
  } catch (error) {
    l84Results.push({ name, ok: false, message: error.message });
  }
}

function l84Expect(actual) {
  return {
    toBe(expected) {
      if (!Object.is(actual, expected)) {
        throw new Error(`expected ${JSON.stringify(expected)}, got ${JSON.stringify(actual)}`);
      }
    },
    toEqual(expected) {
      if (JSON.stringify(actual) !== JSON.stringify(expected)) {
        throw new Error(`expected ${JSON.stringify(expected)}, got ${JSON.stringify(actual)}`);
      }
    },
    toThrow(message) {
      let threw = null;
      try { actual(); } catch (error) { threw = error; }
      if (!threw) throw new Error("expected the function to throw, and it did not");
      if (message && !threw.message.includes(message)) {
        throw new Error(`expected the error to mention "${message}", got "${threw.message}"`);
      }
    },
  };
}

// the code under test
function l84Validate(values) {
  const errors = {};
  if (!values.name?.trim()) errors.name = "Name is required";
  if (!values.email?.includes("@")) errors.email = "That doesn't look like an email";
  return errors;
}

// the tests
l84It("requires a name", () => l84Expect(l84Validate({ name: "", email: "a@b" })).toEqual({ name: "Name is required" }));
l84It("accepts a complete form", () => l84Expect(l84Validate({ name: "Ada", email: "ada@example.com" })).toEqual({}));
l84It("this one is wrong on purpose", () => l84Expect(l84Validate({ name: "Ada", email: "ada@example.com" })).toEqual({ name: "?" }));

for (const result of l84Results) {
  console.log(result.ok ? `  PASS  ${result.name}` : `  FAIL  ${result.name}\n        ${result.message}`);
}
console.log(`\n${l84Results.filter((r) => r.ok).length} passed, ${l84Results.filter((r) => !r.ok).length} failed`);

// `it` is a try/catch. `expect` is a comparison that throws a readable message. Everything else a
// real runner gives you — watch mode, jsdom, coverage, parallel files — is convenience around
// exactly this.

### Exercise

Part 1 is **runnable — plain JS**; parts 2 and 3 are **in your project** (`react-scratch`).

1. Add `toContain`, `toHaveLength` and a `l84Describe(name, body)` that groups results and prints
   them indented under the group name. Re-run the three tests inside a group.
2. Install Vitest in `react-scratch` with the command above, add the `test` script and the `test`
   block in `vite.config.js`, and get `npx vitest run` to report **0 test files found**. That
   message means the setup works and you simply have no tests yet.
3. Write `src/validate.js` and `src/validate.test.js` with two passing cases, then deliberately
   break one — change the expected message — and read the failure output. Write down what it tells
   you that your own runner in part 1 does not.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** Sorting a real list of candidate tests.

Here are ten things someone proposed testing in Mini-project 3. Write `l84Verdict(item)` returning
`"worth it"`, `"not worth it"` or `"test the behaviour instead"`, with a one-line reason:

```
1. the reducer returns the right state for the FILTER_CHANGED action
2. the component calls useState exactly twice
3. searching for "ada" shows only Ada's row
4. the loading spinner has the class "spinner-lg"
5. the empty state appears when the API returns []
6. the service layer sends the right URL for a given query
7. the heading text is "Employee Directory"
8. clicking a row opens the detail panel
9. React Router navigates to /employees/3
10. a fixed bug: selecting a department no longer clears the search box
```

Then answer in comments: numbers 2 and 4 both fail on a refactor that changes nothing a user can
see. What single sentence from this lesson rules them both out — and why is number 9 on the
"not worth it" list even though it looks like behaviour?

In [ ]:
// Your code here

## LESSON 85 — Testing pure logic

Start here, always. Pure functions are the cheapest thing in a codebase to test — no DOM, no
mocking, no async — and in a React app they are where the rules live.

You have written several already: the validator from LESSON 33 and 79, the reducers from LESSON
52 to 54, the selectors and derived values from LESSON 21 and 29, the `cx()` from LESSON 13. All
of them are `(input) → output` with no side effects, which is exactly what a test is good at.

### What a good pure-logic test looks like

Three cases, not thirty:

```js
import { describe, expect, it } from "vitest";
import { counterReducer } from "./counterReducer.js";

describe("counterReducer", () => {
  it("increments", () => {
    expect(counterReducer({ count: 1 }, { type: "incremented" })).toEqual({ count: 2 });
  });

  it("does not mutate the state it was given", () => {
    const state = { count: 1 };
    counterReducer(state, { type: "incremented" });
    expect(state).toEqual({ count: 1 });
  });

  it("throws on an unknown action", () => {
    expect(() => counterReducer({ count: 0 }, { type: "nope" })).toThrow("Unknown action: nope");
  });
});
```

The second test is the one people leave out, and it is the one that catches the bug that actually
happens (LESSON 27 and 52: a reducer must not mutate). The third pins down behaviour you decided
on deliberately.

### Choosing cases

Not "every input". Pick the boundaries:

| pick | for a validator | for a reducer |
|---|---|---|
| the ordinary case | a valid form | the common action |
| the boundary | exactly 18, exactly 100 characters | an empty list, one item |
| the wrong shape | `undefined`, `""`, `null` | an unknown action type |
| the interaction | two rules failing at once | two actions in sequence |

Three to five cases per function is usually right. If a function needs fifteen, that is a finding
about the function, not about the test.

### The two matchers that matter

- **`toBe`** compares with `Object.is` — right for numbers, strings, booleans, and for asking
  "is it the very same object?"
- **`toEqual`** compares structurally — right for objects and arrays.

`expect({ count: 2 }).toBe({ count: 2 })` fails, for exactly the reason `memo` did not skip in
LESSON 69. `toBe` is also how you assert an immutable update **returned a new object**:

```js
const next = reducer(state, action);
expect(next).not.toBe(state);       // a new object…
expect(state).toEqual({ count: 1 }); // …and the old one is untouched
```

### Tests as documentation you cannot ignore

A test file is the only documentation that fails when it becomes untrue. `it("does not mutate the
state it was given")` states a rule about your reducer that a comment could also state — except
that this one is checked on every run.

That is worth remembering when naming: **the `it` string should say the rule, not the mechanics.**
`it("returns { count: 2 }")` describes the assertion; `it("increments")` describes the behaviour,
and still reads correctly after you change the shape of the state.

### Key Notes

- Pure functions first: validators, reducers, selectors, formatters. No DOM, no mocks.
- Always include the "does not mutate" test for a reducer — it is the bug that happens.
- `toBe` for primitives and identity; `toEqual` for structure.
- Name the test after the rule, not after the assertion.

### Example

**Runnable — plain JS.** The functions and the cases. The runner is LESSON 84's twenty-liner, so
this really runs here; in `react-scratch` the same tests run under Vitest with `expect` and `it`
imported instead.

In [ ]:
// L85 — choosing cases for a reducer and a validator

function l85Reducer(state, action) {
  switch (action.type) {
    case "added":
      return { ...state, items: [...state.items, { id: action.id, text: action.text, done: false }] };
    case "toggled":
      return {
        ...state,
        items: state.items.map((item) => (item.id === action.id ? { ...item, done: !item.done } : item)),
      };
    case "cleared":
      return { ...state, items: state.items.filter((item) => !item.done) };
    default:
      throw new Error(`Unknown action: ${action.type}`);
  }
}

// --- the same tiny runner as LESSON 84 --------------------------------------
const l85Log = [];
function l85It(name, body) {
  try { body(); l85Log.push(`  PASS  ${name}`); }
  catch (error) { l85Log.push(`  FAIL  ${name}\n        ${error.message}`); }
}
function l85Expect(actual) {
  const fail = (m) => { throw new Error(m); };
  return {
    toBe: (e) => Object.is(actual, e) || fail(`expected ${JSON.stringify(e)}, got ${JSON.stringify(actual)}`),
    notToBe: (e) => !Object.is(actual, e) || fail("expected a different object, got the same one"),
    toEqual: (e) => JSON.stringify(actual) === JSON.stringify(e) || fail(`expected ${JSON.stringify(e)}, got ${JSON.stringify(actual)}`),
    toThrow: (m) => {
      let threw = null;
      try { actual(); } catch (error) { threw = error; }
      if (!threw) fail("expected it to throw");
      if (m && !threw.message.includes(m)) fail(`expected "${m}", got "${threw.message}"`);
    },
  };
}

const l85Start = { items: [{ id: 1, text: "write tests", done: false }] };

// the ordinary case
l85It("adds an item", () => {
  const next = l85Reducer(l85Start, { type: "added", id: 2, text: "run them" });
  l85Expect(next.items.length).toBe(2);
  l85Expect(next.items[1].text).toBe("run them");
});

// the rule that actually breaks
l85It("does not mutate the state it was given", () => {
  const next = l85Reducer(l85Start, { type: "added", id: 2, text: "run them" });
  l85Expect(l85Start.items.length).toBe(1);
  l85Expect(next).notToBe(l85Start);
});

// the boundary
l85It("clears nothing when nothing is done", () => {
  l85Expect(l85Reducer(l85Start, { type: "cleared" }).items.length).toBe(1);
});

// the decision you made on purpose
l85It("throws on an unknown action", () => {
  l85Expect(() => l85Reducer(l85Start, { type: "nope" })).toThrow("Unknown action: nope");
});

// two actions in sequence — the interaction case
l85It("toggling then clearing removes the item", () => {
  const toggled = l85Reducer(l85Start, { type: "toggled", id: 1 });
  l85Expect(l85Reducer(toggled, { type: "cleared" }).items).toEqual([]);
});

for (const line of l85Log) console.log(line);

### Exercise

Part 1 is **runnable — plain JS**; part 2 is **in your project** (`react-scratch`).

1. Write the five cases for LESSON 79's `validate(values)`: a valid form, one rule failing, two
   rules failing on different fields, an empty object, and `undefined` for a field. One of those
   last two probably crashes your validator — that is the test doing its job, so fix the function,
   not the test.
2. Copy your Mini-project 3 reducer into `react-scratch` and write real Vitest tests for it:
   the ordinary action, the no-mutation rule, one boundary, and the unknown action. Run
   `npx vitest run` and paste the passing output into a comment.
3. Now break the reducer on purpose — use `state.items.push(...)` instead of a spread — and
   confirm which of your tests fails. If none does, your no-mutation test is not written correctly;
   fix it. This is the single most valuable test in the file, so make sure it can actually fail.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

**Runnable — plain JS.** A test that cannot fail is worse than no test, because it reports safety
it does not provide.

Here are four tests. For each, decide whether it can ever fail, and if not, why:

```js
it("adds an item", () => {
  const next = reducer(state, { type: "added", id: 2 });
  expect(next).toBeDefined();
});

it("validates", () => {
  expect(typeof validate({})).toBe("object");
});

it("does not mutate", () => {
  const next = reducer(state, { type: "added", id: 2 });
  expect(next).toEqual(next);
});

it("filters the list", () => {
  const visible = filter(items, "");
  expect(visible.length).toBeGreaterThanOrEqual(0);
});
```

Write `l85CanFail(test)` returning your verdict for each, then rewrite all four into tests that
can fail — and prove it by running each rewritten test against a deliberately broken
implementation.

Then, in a comment: all four share one shape. Name it in a sentence, and say what question you
should ask of every assertion you write to avoid it.

In [ ]:
// Your code here

## LESSON 86 — React Testing Library

Testing a component means rendering it, interacting with it, and asserting what the user would
see. React Testing Library does those three things, and its whole design follows from one
sentence:

> your test should resemble how users interact with your code (component, page, etc.) as much as
> possible

That is why there is no API for reading a component's state, and no way to ask which Hook ran. A
user cannot do those things, so a test should not depend on them.

### The four pieces

```jsx
import { render, screen } from "@testing-library/react";
import userEvent from "@testing-library/user-event";
import { expect, it } from "vitest";
import Counter from "./Counter.jsx";

it("counts clicks", async () => {
  const user = userEvent.setup();
  render(<Counter label="Clicks" />);

  await user.click(screen.getByRole("button", { name: "increment" }));

  expect(screen.getByText("Clicks: 1")).toBeInTheDocument();
});
```

- **`render(<Component />)`** mounts it into the jsdom document.
- **`screen`** queries that document — the same object regardless of what you rendered.
- **`userEvent`** performs realistic interactions. `setup()` first, and every action is `await`ed.
- **the assertion** is about what is on screen.

Note what is absent: no reference to the component instance, no state inspection, no snapshot of
the markup. Read the test and you learn what the component *does*.

### Finding things the way a user would

The query priority is a ranking, not a menu, and `getByRole` is the *"top preference for just
about everything"*:

| priority | queries | when |
|---|---|---|
| 1 — accessible to everyone | `getByRole`, `getByLabelText`, `getByPlaceholderText`, `getByText`, `getByDisplayValue` | almost always; `getByLabelText` for form fields |
| 2 — semantic | `getByAltText`, `getByTitle` | when there is no role or label |
| 3 — test ids | `getByTestId` | last resort, when nothing user-visible identifies it |

`getByRole("button", { name: "increment" })` finds the element **the way a screen reader would** —
by its role and its accessible name. A test written this way fails when the button stops being
reachable, which is a real failure and one no other test catches.

There is a real benefit here beyond tidiness: if you cannot find an element by role or label, that
is usually a signal that it is not accessible to a screen reader either.

### `getBy`, `queryBy`, `findBy`

Three prefixes, three jobs, and using the wrong one is the most common mistake in this lesson:

| prefix | returns | use it to |
|---|---|---|
| `getBy…` | the node, **throws** if not found | assert something **is** there |
| `queryBy…` | the node, or **`null`** | assert something is **not** there |
| `findBy…` | a **Promise**, retrying until it appears | wait for something async |

```jsx
expect(screen.queryByRole("status")).not.toBeInTheDocument();   // not there yet
expect(await screen.findByRole("status")).toBeInTheDocument();  // appears after loading
```

`expect(screen.getByRole("status")).not.toBeInTheDocument()` can never pass — `getBy` throws
before `expect` is reached. That is why `queryBy` exists.

### The failure messages are the feature

Testing Library prints the DOM when a query fails, and the accessible roles when a role query
fails. Real output, from a failing test written for this lesson:

```text
TestingLibraryElementError: Unable to find an accessible element with the role "link" and name "increment"

Here are the accessible roles:

  button:

  Name "increment":
  <button />
```

It has told you the role you should have asked for. Read the message before changing the test.

### What to test, and what to leave

Test the behaviour that matters: an interaction produces a change, a conditional element appears,
a form submits its values, the empty state renders for an empty list, an error message appears
when the request fails.

Do not test that a heading contains a particular string, that a class name is present, or that a
child component received a particular prop. Those pass and fail for reasons unrelated to whether
the app works.

### Key Notes

- Render, query with `screen`, interact with `userEvent`, assert what the user would see.
- `getByRole` first; `getByTestId` last. If nothing user-visible identifies an element, that is
  a finding.
- `getBy` throws, `queryBy` returns `null` (use it for absence), `findBy` returns a Promise (use
  it for async).
- `userEvent.setup()` once per test, and `await` every interaction.

### Example

**In your project — `react-scratch`.** No notebook cell and no playground: this needs a real
component, a real DOM and the packages from LESSON 84.

Create `src/Counter.jsx`:

```jsx
import { useState } from "react";

export default function Counter({ label = "Count" }) {
  const [count, setCount] = useState(0);
  return (
    <div>
      <p>{label}: {count}</p>
      <button onClick={() => setCount(count + 1)}>increment</button>
      {count >= 3 && <p role="status">that is plenty</p>}
    </div>
  );
}
```

and `src/Counter.test.jsx`:

```jsx
import { render, screen } from "@testing-library/react";
import userEvent from "@testing-library/user-event";
import { expect, it } from "vitest";
import Counter from "./Counter.jsx";

it("starts at zero", () => {
  render(<Counter />);
  expect(screen.getByText("Count: 0")).toBeInTheDocument();
});

it("counts clicks", async () => {
  const user = userEvent.setup();
  render(<Counter label="Clicks" />);

  await user.click(screen.getByRole("button", { name: "increment" }));
  expect(screen.getByText("Clicks: 1")).toBeInTheDocument();
});

it("shows a message only after three clicks", async () => {
  const user = userEvent.setup();
  render(<Counter />);

  expect(screen.queryByRole("status")).not.toBeInTheDocument();

  const button = screen.getByRole("button", { name: "increment" });
  await user.click(button);
  await user.click(button);
  await user.click(button);

  expect(await screen.findByRole("status")).toHaveTextContent("that is plenty");
});
```

`npx vitest run` should report three passing tests. All three describe things a user can do and
see, and none of them mentions `useState`.

### Exercise

**In your project — `react-scratch`.**

1. Get the three tests above passing. Then delete `role="status"` from the component and read the
   failure. What does Testing Library print, and what does it suggest you query instead?
2. Add a fourth test that fails on purpose — assert the count is 5 after one click — and read the
   printed DOM. Write down, in a comment, why that output makes `getByTestId` unnecessary most of
   the time.
3. Write a test for a **controlled input** (LESSON 30): render a small form, use
   `await user.type(screen.getByLabelText("Name"), "Ada")`, and assert the value. Then remove the
   `<label>`'s association with the input and watch `getByLabelText` fail — and say what that
   tells you about the component, not about the test.
4. Test one of your Mini-project 3 components end to end: render it with fixed props, interact,
   and assert what changes. Then answer: did you have to change the component to make it testable?
   If so, was the change an improvement anyway? (It usually is — that is LESSON 66's separation
   arriving from a different direction.)

_Don't open `solutions.ipynb` until you've actually tried._

### Mini challenge

**In your project — `react-scratch`.** Testing the four states (LESSON 46) is where component
tests earn their keep, because the empty and error branches are the ones nobody clicks through by
hand.

Build a small `<Employees />` that takes a `load` function as a prop, calls it in an Effect, and
renders loading / error / empty / success. Passing the loader as a prop is the whole trick: the
test supplies it, so there is no network and no mocking library.

Write four tests, one per state, using a loader that resolves late, rejects, resolves `[]`, and
resolves two rows. You will need `findBy…` for at least two of them.

Then answer in comments:

1. Which of the four did you get wrong first, and what did the failure output tell you?
2. Passing `load` as a prop made this testable without any mocking. Which lesson does that
   decision come from, and what would you have needed instead if the component imported the
   service directly?
3. One of these four tests would have caught a real bug in your Mini-project 3. Which, and why did
   you not notice the bug by hand?

> **Topic 26 complete — LESSON 84, 85 and 86.** What is worth testing, Vitest reading the Vite
> config you already have, pure logic first, and components tested the way a user meets them.
>
> Everything here happened in `react-scratch`, so the playground is still React and Vite only —
> and Mini-project 4 asks for a few tests over exactly the things this topic says are worth it:
> reducers and validators.